# Delta Lake Lakehouse – Starter Notebook

This notebook walks through the **bronze → silver → gold** medallion pattern using Delta Lake on MinIO.

| Layer  | S3A path                          | Purpose                        |
|--------|-----------------------------------|--------------------------------|
| Bronze | `s3a://lakehouse/bronze/`         | Raw, immutable ingestion       |
| Silver | `s3a://lakehouse/silver/`         | Cleaned & deduplicated         |
| Gold   | `s3a://lakehouse/gold/`           | Aggregated, analytics-ready    |

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("DeltaLakehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-424a739a-7a8d-4ba8-b6d4-32db34a2708d;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.13/4.0.0/delta-spark_2.13-4.0.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.13;4.0.0!delta-spark_2.13.jar (765ms)
downloading https://repo1.maven.org/mav

Spark version: 4.0.1


## 1 – Bronze: ingest raw data

In [2]:
from pyspark.sql import Row
import datetime

raw_data = [
    Row(id=1, name="Alice",   amount=120.5,  ts="2024-01-01"),
    Row(id=2, name="Bob",     amount=200.0,  ts="2024-01-02"),
    Row(id=3, name="Alice",   amount=95.0,   ts="2024-01-03"),
    Row(id=2, name="Bob",     amount=200.0,  ts="2024-01-02"),   # duplicate
    Row(id=4, name="Charlie", amount=-50.0,  ts="2024-01-04"),   # bad record
]

bronze_path = "s3a://lakehouse/bronze/transactions"

(
    spark.createDataFrame(raw_data)
    .write.format("delta")
    .mode("append")
    .save(bronze_path)
)

spark.read.format("delta").load(bronze_path).show()

26/04/08 15:49:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+-------+------+----------+
| id|   name|amount|        ts|
+---+-------+------+----------+
|  3|  Alice|  95.0|2024-01-03|
|  2|    Bob| 200.0|2024-01-02|
|  4|Charlie| -50.0|2024-01-04|
|  1|  Alice| 120.5|2024-01-01|
|  2|    Bob| 200.0|2024-01-02|
+---+-------+------+----------+



## 2 – Silver: clean & deduplicate

In [3]:
silver_path = "s3a://lakehouse/silver/transactions"

silver_df = (
    spark.read.format("delta").load(bronze_path)
    .filter("amount > 0")           # drop bad records
    .dropDuplicates(["id", "ts"])   # remove exact duplicates
)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .save(silver_path)
)

spark.read.format("delta").load(silver_path).show()

+---+-----+------+----------+
| id| name|amount|        ts|
+---+-----+------+----------+
|  1|Alice| 120.5|2024-01-01|
|  2|  Bob| 200.0|2024-01-02|
|  3|Alice|  95.0|2024-01-03|
+---+-----+------+----------+



## 3 – Gold: aggregate for analytics

In [4]:
from pyspark.sql import functions as F

gold_path = "s3a://lakehouse/gold/customer_summary"

gold_df = (
    spark.read.format("delta").load(silver_path)
    .groupBy("name")
    .agg(
        F.count("id").alias("tx_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
    )
)

(
    gold_df.write.format("delta")
    .mode("overwrite")
    .save(gold_path)
)

spark.read.format("delta").load(gold_path).show()

+-----+--------+------------+----------+
| name|tx_count|total_amount|avg_amount|
+-----+--------+------------+----------+
|  Bob|       1|       200.0|     200.0|
|Alice|       2|       215.5|    107.75|
+-----+--------+------------+----------+



## 4 – Delta Lake features: time travel & history

In [5]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, bronze_path)

print("=== Table History ===")
dt.history().select("version", "timestamp", "operation").show(truncate=False)

print("=== Version 0 (time travel) ===")
spark.read.format("delta").option("versionAsOf", 0).load(bronze_path).show()

=== Table History ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|0      |2026-04-08 15:49:34|WRITE    |
+-------+-------------------+---------+

=== Version 0 (time travel) ===
+---+-------+------+----------+
| id|   name|amount|        ts|
+---+-------+------+----------+
|  3|  Alice|  95.0|2024-01-03|
|  2|    Bob| 200.0|2024-01-02|
|  4|Charlie| -50.0|2024-01-04|
|  1|  Alice| 120.5|2024-01-01|
|  2|    Bob| 200.0|2024-01-02|
+---+-------+------+----------+



## 5 – Upsert (MERGE) example

In [6]:
updates = spark.createDataFrame([
    Row(id=2, name="Bob",   amount=250.0,  ts="2024-01-02"),   # update
    Row(id=5, name="Diana", amount=300.0,  ts="2024-01-05"),   # insert
])

(
    DeltaTable.forPath(spark, silver_path)
    .alias("target")
    .merge(updates.alias("src"), "target.id = src.id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

spark.read.format("delta").load(silver_path).orderBy("id").show()

26/04/08 15:51:48 WARN MapPartitionsRDD: RDD 144 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
                                                                                

+---+-----+------+----------+
| id| name|amount|        ts|
+---+-----+------+----------+
|  1|Alice| 120.5|2024-01-01|
|  2|  Bob| 250.0|2024-01-02|
|  3|Alice|  95.0|2024-01-03|
|  5|Diana| 300.0|2024-01-05|
+---+-----+------+----------+

